# Rozklad variability ve vyřizování pojistných plnění napříč organizační hierarchií pojišťovny pomocí PROC NESTED

## Shrnutí

Pojišťovna majetkového a odpovědnostního pojištění chce vědět, **kde**
vzniká nekonzistence ve vyřizování pojistných plnění: je způsobena
rozdíly mezi geografickými regiony, mezi pobočkami, mezi jednotlivými
likvidátory, nebo prostě jen šumem mezi jednotlivými škodami? Tento
notebook sestaví syntetickou, plně vnořenou knihu dat o škodách z
autopojištění (region > pobočka > likvidátor > škoda) a pomocí
**PROC NESTED** provede analýzu rozptylu s náhodnými efekty, přičemž
odhadne komponentu rozptylu příslušející každé úrovni hierarchie a
uvede ji jako procento z celku. Výsledek řekne vedení, na kterou
organizační úroveň se zaměřit při zlepšování konzistence vyřizování
plnění.

## Zdroje dat

Všechna data jsou generována přímo v prvním kroku DATA — žádné
externí soubory, žádná síť. Návrh je **zcela vnořený**: každá pobočka
patří přesně k jednomu regionu, každý likvidátor přesně k jedné
pobočce a každá škoda přesně k jednomu likvidátorovi.

| Datová sada | Řádky | Proměnná | Typ | Role | Popis |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (úroveň 1, nejvyšší) | Geografický region (5 regionů: SV, JV, SZ, ZP, JZ) |
| | | `Branch` | char | CLASS (úroveň 2, vnořená v Region) | Kód pobočky (2 pobočky na region) |
| | | `Adjuster` | char | CLASS (úroveň 3, vnořená v Branch) | ID likvidátora škod (2 likvidátoři na pobočku) |
| | | `Settlement` | num | VAR (závislá) | Vyplacené pojistné plnění za škodu z autopojištění, v USD |
| | | `CycleDays` | num | VAR (závislá) | Počet dnů od prvního oznámení škody do jejího vyřízení |

Syntetická struktura: 5 regionů x 2 pobočky x 2 likvidátoři x 2 škody
= 40 pozorování. Náhodný efekt regionu, náhodný efekt pobočky vnořené
v regionu, náhodný efekt likvidátora vnořeného v pobočce a reziduum na
úrovni škody jsou vrstveny aditivně pomocí `rand('NORMAL', ...)`, takže
komponenty rozptylu lze zpětně zjistit. Efekt regionu je vygenerován s
největší směrodatnou odchylkou (2200), takže *kde* je škoda vyřizována
je hlavním hybatelem. Seed je pevně nastaven pomocí
`call streaminit(20260531)`. Kompaktní, plně vyvážený návrh udržuje
každou úroveň hierarchie obsazenou se skutečnými stupni volnosti.

# Rozklad variability ve vyřizování pojistných plnění pomocí PROC NESTED

Když pojišťovna majetkového a odpovědnostního pojištění posuzuje, kolik
platí za vyřízení škod z autopojištění, často zjistí velkou variabilitu.
Část této variability je nevyhnutelná (každá nehoda je jiná), ale část
odráží **organizační** faktory — jeden region je štědřejší než jiný,
pobočka je velkorysejší, jednotlivý likvidátor je odlehlý.

Data mají přísně **vnořenou** (hierarchickou) strukturu:

```
Region  ->  Pobočka (vnořená v regionu)  ->  Likvidátor (vnořený v pobočce)  ->  jednotlivé škody
```

Pobočka patří přesně k jednomu regionu a likvidátor přesně k jedné
pobočce, takže faktory jsou vnořené, nikoli zkřížené. `PROC NESTED`
provádí pro přesně takový návrh analýzu rozptylu s náhodnými efekty a
odhaduje **komponentu rozptylu** na každé úrovni, vyjádřenou jako
procento z celku. Toto procentuální rozdělení odpovídá na obchodní
otázku: *na kterou vrstvu organizace bychom se měli zaměřit, aby bylo
vyřizování škod konzistentnější?*

## Krok 1 — Generování syntetické, plně vnořené knihy škod

Simulujeme 5 regionů, každý se 2 pobočkami, každá se 2 likvidátory,
každý vyřizující 2 škody (celkem 40 řádků). Odezvu sestavíme z
aditivních náhodných efektů, aby každá úroveň skutečně přispívala k
rozptylu:

- efekt **regionu** (největší rozptyl — regiony se liší nejvíce),
- efekt **pobočky vnořené v regionu**,
- efekt **likvidátora vnořeného v pobočce**,
- a **reziduum na úrovni škody** (šum uvnitř likvidátora).

`call streaminit` pevně nastavuje seed pro reprodukovatelnost; každý
efekt je generován pomocí `rand('NORMAL', mean, sd)`. Efekt regionu
používá nejširší směrodatnou odchylku, takže by měl nést největší podíl
celkového rozptylu. Generujeme také druhou odezvu, `CycleDays`, která
sdílí stejnou hierarchii, takže později můžeme ukázat analýzu více
odezev.

In [1]:
data claims;
   CALL streaminit(20260531);
   DÉLKA Region $2 Branch $6 Adjuster $9;

   /* Náhodný efekt na úrovni regionu: regiony se liší nejvíce. */
   OPAKUJ r = 1 TO 5;
      Region = SCAN('SV JV SZ ZP JZ', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Pobočka vnořená v regionu. */
      OPAKUJ b = 1 TO 2;
         Branch = catx('-', Region, ZAPSAT(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Likvidátor vnořený v pobočce. */
         OPAKUJ a = 1 TO 2;
            Adjuster = catx('-', Branch, ZAPSAT(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Jednotlivé škody vyřizované tímto likvidátorem. */
            OPAKUJ claim = 1 TO 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               KDYŽ Settlement < 0 PAK Settlement = 0;
               KDYŽ CycleDays  < 1 PAK CycleDays  = 1;
               VÝSTUP;
            KONEC;
         KONEC;
      KONEC;
   KONEC;

   PONECHAT Region Branch Adjuster Settlement CycleDays;
SPUSTIT;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Krok 2 — Seřazení podle hierarchie vnoření

`PROC NESTED` vyžaduje, aby vstupní data byla **seřazena podle
proměnných CLASS v pořadí, v jakém budou uvedeny** — nejvyšší faktor
jako první. Řadíme podle `Region Branch Adjuster`, aby procedura mohla
hierarchii správně projít.

In [2]:
PROCEDURA ŘADIT data=claims;
   PODLE Region Branch Adjuster;
SPUSTIT;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Krok 3 — Analýza komponent rozptylu výše plnění

Jádro analýzy. Proměnné CLASS uvádíme od nejvyšší po nejnižší úroveň
(`Region Branch Adjuster`); nejnižší úroveň replikace — jednotlivé
škody — je automaticky považována za chybový člen. `VAR Settlement`
pojmenovává jedinou odezvu.

Příkaz `LABEL` poskytuje přívětivější zobrazovaný název odezvy v
záhlaví výstupu. Výstup vytváří koeficienty očekávaných středních
čtverců, tabulku analýzy rozptylu (s F-testem každé komponenty
rozptylu proti nule) a odhad **komponenty rozptylu** na každé úrovni
spolu s jejím **procentem z celku**.

In [3]:
NÁZEV 'Komponenty rozptylu vyplacených pojistných plnění';
title2 'ANOVA náhodných efektů: region / pobočka / likvidátor';

PROCEDURA nested data=claims;
   TŘÍDA Region Branch Adjuster;
   PROMĚNNÁ Settlement;
   ŠTÍTEK Settlement = 'Vyplacené plnění (USD)';
SPUSTIT;


                                   Komponenty rozptylu vyplacených pojistných plnění                                    
                                 ANOVA náhodných efektů: region / pobočka / likvidátor                                  


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Vyplacené plnění (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826


NOTE: Option TITLE changed to Komponenty rozptylu vyplacených pojistných plnění.
NOTE: Option TITLE2 changed to ANOVA náhodných efektů: region / pobočka / likvidátor.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Krok 4 — Analýza dvou odezev najednou

Vedení také zajímá **doba vyřízení** — jak dlouho trvá vyřízení škod.
Přidáme `CycleDays` do seznamu `VAR`. Při více než jedné závislé
proměnné `PROC NESTED` navíc vypisuje **analýzu kovariace**, která
ukazuje, jak spolu obě odezvy souvisí na každé úrovni hierarchie
(například zda regiony, které platí více, škody také vyřizují
pomaleji).

In [4]:
NÁZEV 'Komponenty rozptylu plnění a doby vyřízení';
title2 'Dvě odezvy napříč stejnou vnořenou hierarchií';

PROCEDURA nested data=claims;
   TŘÍDA Region Branch Adjuster;
   PROMĚNNÁ Settlement CycleDays;
   ŠTÍTEK Settlement = 'Vyplacené plnění (USD)'
         CycleDays  = 'Počet dnů do vyřízení';
SPUSTIT;


                                       Komponenty rozptylu plnění a doby vyřízení                                       
                                     Dvě odezvy napříč stejnou vnořenou hierarchií                                      


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Vyplacené plnění (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826


NOTE: Option TITLE changed to Komponenty rozptylu plnění a doby vyřízení.
NOTE: Option TITLE2 changed to Dvě odezvy napříč stejnou vnořenou hierarchií.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Krok 5 — Pohled pouze na rozptyl pomocí volby AOV

Pro rychlý souhrn komponent rozptylu napříč oběma odezvami volba `AOV`
omezí výstup na statistiky analýzy rozptylu a **potlačí část analýzy
kovariace**. Toto je kompaktní pohled, který by portfoliový analytik
prohlížel, když potřebuje jen rozklad rozptylu po úrovních pro každou
odezvu, ne kovarianci mezi odezvami.

In [5]:
NÁZEV 'Pouze komponenty rozptylu (AOV)';

PROCEDURA nested data=claims aov;
   TŘÍDA Region Branch Adjuster;
   PROMĚNNÁ Settlement CycleDays;
   ŠTÍTEK Settlement = 'Vyplacené plnění (USD)'
         CycleDays  = 'Počet dnů do vyřízení';
SPUSTIT;

NÁZEV;


                                            Pouze komponenty rozptylu (AOV)                                             
                                     Dvě odezvy napříč stejnou vnořenou hierarchií                                      


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Vyplacené plnění (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826


NOTE: Option TITLE changed to Pouze komponenty rozptylu (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interpretace výsledků

Sloupec **Percent of Total** (procento z celku) v tabulce analýzy
rozptylu je tím hlavním údajem. Čtení odshora dolů udává podíl na
celkové variabilitě vyplacených plnění připadající na každou
organizační vrstvu. Pro výši plnění běh přináší:

- **Region — 54.4%** (Pr > F = 0.0304). Data byla vygenerována s
  největším rozptylem na úrovni regionu a analýza jej odhaluje: více
  než polovina veškeré variability plnění je *mezi* regiony, statisticky
  významný důkaz, že *kde* je škoda vyřizována, je hlavním hybatelem.
- **Pobočka (v rámci regionu) — 9.0%** (Pr > F = 0.18). Skromný
  dodatečný podíl při přechodu z regionu na jednotlivé pobočky; zde
  není významný.
- **Likvidátor (v rámci pobočky) — 3.7%** (Pr > F = 0.33). Jen málo
  variability lze přisoudit jednotlivému likvidátorovi v tomto portfoliu.
- **Chyba — 32.9%.** Reziduální šum mezi jednotlivými škodami uvnitř
  likvidátora; toto je neredukovatelná variabilita, kterou žádná
  organizační páka neodstraní.

Každá úroveň nese také **F-test (Pr > F)** nulové hypotézy, že její
komponenta rozptylu je nulová. Malá p-hodnota regionu (0.0304) je
statistickým důkazem skutečných systematických rozdílů mezi regiony,
nikoli náhody výběru.

Odezva doby vyřízení vypráví paralelní příběh: **Region tvoří 45.8%**
variability počtu dnů do vyřízení (Pr > F = 0.0448, opět významné),
přičemž pobočka a likvidátor přispívají jednociferným podílem a
reziduum nese 40.1%. Jak **kolik** pojišťovna platí, tak **jak dlouho**
to trvá, je tedy řízeno především regionem.

Běh se dvěma odezvami přidává **analýzu kovariace**. Zde úroveň
regionu řídí křížové součiny a celkový **korelační koeficient je
-0.36** — negativní vztah: regiony, které vyplácejí vyšší plnění, mají
tendenci je **vyřizovat rychleji**, nikoli pomaleji. To je užitečné,
neintuitivní zjištění — nákladné regiony nejsou ty pomalé.

**Obchodní závěr:** protože region dominuje rozkladu procenta z celku u
obou odezev, měla by pojišťovna nejprve standardizovat pokyny pro
vyřizování plnění a limity pravomocí *napříč regiony* — tam se nachází
největší, statisticky významná nekonzistence — než investovat do
zásahů na úrovni poboček či likvidátorů. Záporná korelace mezi plněním
a dobou vyřízení znamená, že neexistuje jediný region, který by byl
zároveň nejnákladnější i nejpomalejší, takže oba problémy lze řešit
samostatnými, na region zaměřenými programy. PROC NESTED mění vágní
pocit, že „naše vyřizování škod je nekonzistentní", v akční,
vrstvu-po-vrstvě popsanou atribuci toho, kde tato nekonzistence
existuje.